In [0]:
# # Spark optimizations for large MERGE operations
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.databricks.photon.sort.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.databricks.adaptive.autoOptimizeShuffle.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

In [0]:
spark.sql("""
    MERGE INTO prod.detection.commercial_id_external_firehose AS target
    USING dev.mohit_gangwani.duplicate_ads_mapping_table AS source
    ON target.fk_commercial_id = source.current_bad_comm_id
    AND target.fk_client_id = source.fk_client_id
    WHEN MATCHED THEN UPDATE SET fk_commercial_id = source.new_correct_id;
    """
)

In [0]:
spark.sql("""
    MERGE INTO prod.detection.commercial_id_external_firehose AS target
    USING dev.mohit_gangwani.duplicate_ads_deletion_table AS source
    ON target.fk_commercial_id = source.current_bad_comm_id
    AND target.fk_client_id = source.fk_client_id
    WHEN MATCHED THEN DELETE;
    """
)

In [0]:
from datetime import datetime, timedelta

# Iterate one week at a time over the last 2 years
start_date = datetime.now() - timedelta(days=730)
end_date = datetime.now()
current = start_date

In [0]:
while current < end_date:
    date_str = current.strftime('%Y-%m-%d')
    spark.sql(
        f"""
        DELETE FROM prod.detection.viewing_commercials_dedup_golden
            WHERE fk_commercial_source_id = 2
            AND commercial_client != 'kinetiq'
            AND DATE(session_start_hour) = '{date_str}';
        """)

    spark.sql(
        f"""
        DELETE FROM prod.detection.viewing_commercials_dedup_golden
            WHERE fk_commercial_source_id = 3
            AND commercial_client != 'SpringServe-Prod'
            AND DATE(session_start_hour) = '{date_str}';
        """)

    spark.sql(
        f"""
        DELETE FROM prod.detection_onn.viewing_commercials_dedup_golden
            WHERE fk_commercial_source_id = 2
            AND commercial_client != 'kinetiq'
            AND DATE(session_start_hour) = '{date_str}';
        """)

    spark.sql(
        f"""
        DELETE FROM prod.detection_onn.viewing_commercials_dedup_golden
            WHERE fk_commercial_source_id = 3
            AND commercial_client != 'SpringServe-Prod'
            AND DATE(session_start_hour) = '{date_str}';
        """)

    spark.sql(f"""
        DELETE FROM prod.detection.viewing_commercials_dedup_golden
        WHERE fk_commercial_id IN (
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
            UNION
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
        )
        AND commercial_client NOT IN ('kinetiq', 'SpringServe-Prod')
        AND DATE(session_start_hour) = '{date_str}';
    """)

    spark.sql(f"""
        DELETE FROM prod.detection.viewing_commercials_golden
        WHERE fk_commercial_id IN (
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
            UNION
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
        )
        AND commercial_client NOT IN ('kinetiq', 'SpringServe-Prod')
        AND DATE(session_start_hour) = '{date_str}';
    """)
    
    spark.sql(f"""
        DELETE FROM prod.detection_onn.viewing_commercials_dedup_golden
        WHERE fk_commercial_id IN (
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
            UNION
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
        )
        AND commercial_client NOT IN ('kinetiq', 'SpringServe-Prod')
        AND DATE(session_start_hour) = '{date_str}';
    """)

    spark.sql(f"""
        DELETE FROM prod.detection_onn.viewing_commercials_golden
        WHERE fk_commercial_id IN (
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
            UNION
            SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
        )
        AND commercial_client NOT IN ('kinetiq', 'SpringServe-Prod')
        AND DATE(session_start_hour) = '{date_str}';
    """)

    print(f"Finished {date_str}")

    current = current + timedelta(days=1)